# Lekcija 11 - Protokol med agenti (A2A)


## Namestitev


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kaj je A2A protokol?

The **Agent-to-Agent (A2A) protokol** je odprt standard, ki omogoča, da si AI agenti medsebojno komunicirajo, odkrijejo drug drugega in sodelujejo — tudi kadar so zgrajeni na različnih okvirjih ali gostovani pri različnih storitvah.

Key concepts:

- **Odkritje** – Agenti objavijo *kartico agenta*, ki opisuje njihove zmožnosti, zaradi česar je drugim agentom (ali orkestratorjem) enostavno najti pravega specialista za nalogo.
- **Posredovanje sporočil** – Agenti izmenjujejo strukturirana sporočila prek skupnega protokola, tako da je zahteva enega agenta razumljena in izpolnjena s strani drugega ne glede na njegovo notranjo implementacijo.
- **Življenjski cikel naloge** – A2A opredeljuje stanja, kot so *submitted*, *working*, *completed* in *failed*, kar orkestratorju omogoča popoln vpogled v to, kako napreduje delegirana naloga.

V tej lekciji simuliramo sodelovanje v slogu A2A tako, da povežemo tri specializirane potovalne agente v potek dela, kjer vsak agent prispeva svoje strokovno znanje in posreduje rezultate naslednjemu.


## Ustvarjanje specializiranih potovalnih agentov


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Sodelovanje več agentov prek poteka dela

Povežemo tri agente v zaporedni potek dela, ki odraža A2A prenašanje sporočil:

1. **CurrencyExchangeAgent** sprejme uporabnikovo zahtevo in zagotovi navodila glede valut.
2. **ActivityPlannerAgent** prejme obogaten kontekst in doda priporočila za aktivnosti.
3. **TravelManagerAgent** združi oba vnosa v končno potovalno poročilo.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Razumevanje A2A v produkciji

V produkcijskem okolju protokol A2A omogoča zmogljive scenarije med storitvami:

| Capability | Description |
|---|---|
| **Medokvirna interoperabilnost** | Agent, zgrajen v enem okviru, lahko delegira naloge agentu, zgrajenemu v katerem koli drugem okviru, združljivem z A2A, kar omogoča pravo interoperabilnost med organizacijami. |
| **Meje storitev** | Agenti lahko delujejo v ločenih mikrostoritvah, oblačnih regijah ali celo različnih organizacijah, pri čemer še vedno brezhibno sodelujejo. |
| **Dinamično odkrivanje** | Orkestrator lahko med izvajanjem poizve register Agent Card, da najde najbolj primernega specialista za določeno podnalogo. |
| **Pretakanje & push notifications** | A2A podpira Server-Sent Events (SSE) za posodobitve napredka v realnem času in push notifications za dolgotrajne naloge. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## Povzetek

V tej lekciji ste se naučili:

1. **Kaj je protokol A2A** — odprt standard za odkrivanje med agenti, pošiljanje sporočil in upravljanje opravil.
2. **Kako ustvariti specializirane agente** — agent za menjavo valut, agent za načrtovanje dejavnosti in orkestrator upravljanja potovanj.
3. **Kako povezati agente v delovni tok** — z uporabo `WorkflowBuilder` za modeliranje zaporednega posredovanja sporočil med agenti.
4. **Kako A2A deluje v produkciji** — omogoča sodelovanje med različnimi ogrodji in storitvami z dinamičnim odkrivanjem in pretočnimi posodobitvami.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Izjava o omejitvi odgovornosti**:
Ta dokument je bil preveden z uporabo storitve za prevajanje z umetno inteligenco [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, upoštevajte, da lahko avtomatizirani prevodi vsebujejo napake ali netočnosti. Izvirni dokument v izvirnem jeziku velja za avtoritativni vir. Za ključne informacije priporočamo strokovni človeški prevod. Ne prevzemamo odgovornosti za morebitne nesporazume ali napačne razlage, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
